In [2]:
import torch
import tiktoken

In [3]:
# We are going to load a model that we trained on the Colab

In [ ]:
model_path = "../_assets/trained_model/m05_base_model_multi_attenion_head_causal.pt"
checkpoint = torch.load(model_path, map_location=torch.device('cpu'))
print(checkpoint.keys())

/var/folders/st/hxtkk4cx271gx3lmqdmgkfpc0000gn/T/ipykernel_87441/2667254455.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_locat

dict_keys(['model_state_dict', 'optimizer_state_dict'])


In [9]:
tokenizer_gpt2 = tiktoken.get_encoding("gpt2")

In [10]:
CONTEXT_LENGTH = 256
EMB_SIZE = 256
VOCAB_SIZE = tokenizer_gpt2.n_vocab
DEVICE = "cpu"

In [11]:
class CausalMultiAttentionHead(torch.nn.Module):
    def __init__(
        self,
        emb_size,
        hidden_dims,
        num_heads=2,
        contex_len=CONTEXT_LENGTH,
        qkv_bias=False,
    ):
        super().__init__()
        assert (
            hidden_dims % num_heads == 0
        ), "Please make sure hidden dims are divisble by num_heads"
        self.num_heads = num_heads
        self.head_dim = hidden_dims // num_heads
        self.WQ = torch.nn.Linear(emb_size, hidden_dims, bias=qkv_bias)
        self.WK = torch.nn.Linear(emb_size, hidden_dims, bias=qkv_bias)
        self.WV = torch.nn.Linear(emb_size, hidden_dims, bias=qkv_bias)
        self.register_buffer(
            "mask", torch.triu(torch.ones(contex_len, contex_len), diagonal=1)
        )

    def forward(self, x):
        batch_size, num_tokens, emb_size = x.shape
        Q = self.WQ(x)
        K = self.WK(x)
        V = self.WV(x)
        # print(Q.shape, K.shape, V.shape)

        Qh = Q.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        Kh = K.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        Vh = V.view(batch_size, num_tokens, self.num_heads, self.head_dim)
        # print(Qh.shape, Kh.shape, Vh.shape)

        Qh = Qh.transpose(1, 2)
        Kh = Kh.transpose(1, 2)
        Vh = Vh.transpose(1, 2)
        # print(Qh.shape, Kh.shape, Vh.shape)
        # print(Qh.shape, Kh.transpose(2, 3).shape, Vh.shape)

        attens = Qh @ Kh.transpose(2, 3)
        attens.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        atten_norm = torch.softmax(attens / emb_size**0.5, dim=-1)
        # print(atten_norm)
        context_vector_h = (atten_norm @ Vh).transpose(1, 2)
        context_vector = context_vector_h.contiguous().view(
            batch_size, num_tokens, emb_size
        )
        return context_vector

In [12]:
class BasicModelWithPosEmbMultiHeadAttenCausal(torch.nn.Module):
    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        emb_size=EMB_SIZE,
        hidden_dim=EMB_SIZE,
    ):
        super().__init__()
        self.emb_layer = torch.nn.Embedding(vocab_size, emb_size)
        self.pos_emb_layer = torch.nn.Embedding(vocab_size, emb_size)
        self.attention_head = CausalMultiAttentionHead(hidden_dim, hidden_dim)
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.GELU(),
            torch.nn.Linear(hidden_dim, VOCAB_SIZE),
        )

    def forward(self, x):
        x = self.emb_layer(x)
        x_pos = self.pos_emb_layer(torch.arange(0, x.shape[1]).to(DEVICE))
        x += x_pos
        x = self.attention_head(x)
        return self.layers(x)

In [13]:
model = BasicModelWithPosEmbMultiHeadAttenCausal()
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

BasicModelWithPosEmbMultiHeadAttenCausal(
  (emb_layer): Embedding(50257, 256)
  (pos_emb_layer): Embedding(50257, 256)
  (attention_head): CausalMultiAttentionHead(
    (WQ): Linear(in_features=256, out_features=256, bias=False)
    (WK): Linear(in_features=256, out_features=256, bias=False)
    (WV): Linear(in_features=256, out_features=256, bias=False)
  )
  (layers): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=256, out_features=50257, bias=True)
  )
)

In [14]:
optim = torch.optim.AdamW(model.parameters())
optim.load_state_dict(checkpoint["optimizer_state_dict"])
optim

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0.01
)

In [34]:
test_text = "Suno Shah Nawaz Dost"
temperature = 5
test_text_encoded = tokenizer_gpt2.encode(test_text)
with torch.no_grad():
    for _ in range(20):
        out_logits = model(torch.tensor([test_text_encoded]))
        # last token as next word
        out_logits = out_logits.squeeze()[-1]

        # 1. Top K
        top_k_logits, top_k_pos = torch.topk(out_logits, k=3)
        new_logits = torch.where(
            condition=out_logits < top_k_logits[-1],
            input=torch.tensor(float("-inf")),
            other=out_logits,
        )

        # 2. Temperature Scaling
        out_logits = new_logits / temperature
        out_logits = torch.softmax(out_logits, dim=-1)
        next_token = torch.multinomial(out_logits, num_samples=1).item()

        # 3. Greedy Decoding
        # next_token = torch.argmax(out_logits)

        next_token = [next_token]
        test_text_encoded += next_token
        next_token_decoded = tokenizer_gpt2.decode(next_token)
        test_text += next_token_decoded

print(test_text)

Suno Shah Nawaz Dost ke, tasveer hai," Mrs. Gisburn ki taraf muda ke
